# ergolog — A minimal, ergonomic Python logging wrapper

Run each cell to see the output. Install with `pip install ergolog` or `uv add ergolog`.

In [ ]:
from time import sleep
from ergolog import eg

## Basic Usage

In [ ]:
eg.debug('debug')
eg.info('info')
eg.warning('warning')
eg.error('error')
eg.critical('critical')

## Named Loggers

In [ ]:
log = eg('test')
log.info('named logger')

### Child Loggers

In [ ]:
one = eg('one')
two = one('two')
two.info('child logger')

## Tags

In [ ]:
with eg.tag('tag1'):
    eg.info('one tag')
    with eg.tag('tag2'):
        eg.info('two tags')
    eg.info('one tag again')

### Keyword Tags

In [ ]:
with eg.tag(keyword='tags', comma='multiple'):
    eg.debug('')
    with eg.tag('regular tag'):
        eg.info('')
        with eg.tag(more='keywords'):
            eg.info('')
    eg.debug('')

### Auto-generated IDs

In [ ]:
with eg.tag(job=eg.uid):
    eg.info('first')
    with eg.tag(job=eg.uid):
        eg.info('nested')
    eg.info('first again')

## Counters

In [ ]:
counter = eg.counter()
with eg.tag(step=counter):
    eg.info('start')       # step=0
    counter += 1
    eg.info('middle')      # step=1
    counter += 1
    eg.info('end')         # step=2

### Loop Enumeration

In [ ]:
loops = eg.counter()
with eg.tag(i=loops):
    for item in loops.count(['a', 'b', 'c']):
        eg.info(f'item {item}')

### Accumulation

In [ ]:
total = eg.counter()
with eg.tag(bytes=total):
    total += 1024
    eg.info('chunk')       # bytes=1024
    total += 512
    eg.info('chunk')       # bytes=1536

## Timers

In [ ]:
with eg.timer(lambda t: eg.debug(f'took {t}S')):
    eg.info('before')
    sleep(0.05)
    eg.info('after')

### Timer as Context Manager

In [ ]:
with eg.timer() as t:
    eg.info('before')
    sleep(0.05)
    eg.info('after')

eg.debug(f'took {t} S')

### Laps

Use `.lap()` to take split times without stopping the timer. Returns elapsed seconds as a float.

In [ ]:
with eg.timer() as t:
    sleep(0.02)
    fetch_time = t.lap()      # returns float, timer keeps running
    sleep(0.02)
    process_time = t.lap()    # returns float from start
    eg.debug(f'fetch={fetch_time:.3f}s process={process_time:.3f}s total={t.elapsed:.3f}s')

### Named Laps

Use `.lap('name')` to record a named lap. Named laps are auto-collected by events.

In [ ]:
with eg.timer() as t:
    sleep(0.02)
    t.lap('fetch')       # records lap name + time
    sleep(0.02)
    t.lap('process')     # records lap name + time

eg.info(f'laps: {t.laps}')

### Timers as Tag Values

Timers can be used as keyword tag values, showing live elapsed time on each log line.

In [ ]:
t = eg.timer()
with eg.tag(elapsed=t):
    eg.info('start')
    sleep(0.05)
    eg.info('middle')
    sleep(0.05)
    eg.info('end')

### Timer + Counter as Tags

In [ ]:
t = eg.timer()
counter = eg.counter()
with eg.tag(elapsed=t, step=counter):
    eg.info('step 1')
    counter += 1
    sleep(0.05)
    eg.info('step 2')
    counter += 1
    eg.info('step 3')

## Trace

In [ ]:
@eg.trace()
def my_function(a, b):
    return a + b

my_function(2, 2)

## Wide Events

Wide events accumulate context throughout an operation and emit a **single log line** at the end.

In [ ]:
with eg.event(user='alice', action='checkout') as e:
    e.set(cart={'items': 3, 'total': 9999})
    e.set(payment={'method': 'card'})

### Manual Emit

In [ ]:
e = eg.event(user='bob', action='search')
e.set(query='ergonomics', results=42)
e.emit()

### Outcome Levels

Events default to `INFO`. Use `e.warn()` or `e.error()` to mark the outcome.

In [ ]:
# Success with concern → WARNING
with eg.event(user='alice', action='payment') as e:
    e.warn('used fallback method', attempts=3)

In [ ]:
# Failure → ERROR
try:
    with eg.event(user='charlie', action='failing_op') as e:
        e.set(step='processing')
        raise ValueError('insufficient funds')
except ValueError:
    pass

### Sealed After Emit

In [ ]:
e = eg.event(user='alice')
e.emit()
e.set(ignored='data')  # Ignored, event is sealed

### Capturing Tags

In [ ]:
with eg.tag(request_id='abc123'):
    with eg.event(operation='tagged_op') as e:
        e.set(extra='data')

## Counters in Events

Counters passed to `e.set()` evaluate at emit time — the event shows their final value.

In [ ]:
counter = eg.counter()
with eg.event(op='batch') as e:
    e.set(processed=counter)
    for i in range(3):
        counter += 1

## Timers in Events

Timers passed to `e.set()` evaluate at emit time — the event shows total elapsed.

In [ ]:
with eg.event(op='export') as e:
    t = eg.timer()
    e.set(duration=t)
    sleep(0.05)

## Named Laps in Events

The killer feature for multi-stage operations. Named laps on timers are auto-collected into events.

In [ ]:
t = eg.timer()
with eg.event(op='pipeline') as e:
    e.set(duration=t)
    sleep(0.02)
    t.lap('fetch')
    sleep(0.02)
    t.lap('process')
    sleep(0.02)
    t.lap('save')

### Explicit Lap Values

For full control, use `t.lap()` (unnamed) and pass the result to `e.set()`.

In [ ]:
with eg.event(op='task') as e:
    t = eg.timer()
    sleep(0.02)
    e.set(fetch_time=t.lap())
    sleep(0.02)
    e.set(process_time=t.lap())

## Counter + Timer + Event Together

All three composability primitives working in concert.

In [ ]:
counter = eg.counter()
t = eg.timer()
with eg.event(op='process_batch') as e:
    e.set(duration=t, items=counter)
    for i in range(3):
        sleep(0.01)
        counter += 1
        t.lap(f'item_{i + 1}')